# Bank Marketing — Data Preprocessing

## Описание этапа

**Цель:** подготовить данные для обучения и сравнения Logistic Regression и SVM без утечки информации.

На основе результатов EDA определим способы обработки признаков, разделим данные на обучающую и тестовую выборки и построим единый пайплайн предобработки.

**Основные задачи:**
1. Исключить признаки, недоступные на момент прогнозирования.
2. Разделить данные на Train и Test.
3. Обработать специальные значения и категориальные признаки.
4. Настроить масштабирование и кодирование.
5. Собрать и проверить пайплайн предобработки.

## 1. Загрузка данных

Импорт библиотек и загрузка исходного датасета.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold

In [2]:
df = pd.read_csv('../data/raw/banking.csv')

df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,44,blue-collar,married,basic.4y,unknown,yes,no,cellular,aug,thu,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,0
1,53,technician,married,unknown,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.021,5195.8,0
2,28,management,single,university.degree,no,yes,no,cellular,jun,thu,...,3,6,2,success,-1.7,94.055,-39.8,0.729,4991.6,1
3,39,services,married,high.school,no,no,no,cellular,apr,fri,...,2,999,0,nonexistent,-1.8,93.075,-47.1,1.405,5099.1,0
4,55,retired,married,basic.4y,no,yes,no,cellular,aug,fri,...,1,3,1,success,-2.9,92.201,-31.4,0.869,5076.2,1


In [3]:
df.shape

(41188, 21)

## 2. Подготовка признаков

Будем прогнозировать отклик клиента перед запланированным телефонным звонком.

Исключим `duration`, поскольку продолжительность разговора становится известна только после его завершения. Остальные признаки пока сохраним, предполагая, что их значения доступны на момент прогнозирования.

Целевая переменная `y` уже закодирована значениями `0` и `1`.

In [4]:
X = df.drop(columns=['y', 'duration'])
y = df['y'].copy()

print('X:', X.shape)
print('y:', y.shape)

X: (41188, 19)
y: (41188,)


In [5]:
assert 'y' not in X.columns
assert 'duration' not in X.columns
assert len(X) == len(y)

## 3. Разделение данных

Разделим данные на обучающую (80%) и тестовую (20%) выборки с сохранением соотношения классов.

Тестовую выборку оставим для итоговой оценки моделей.

### 3.1. Train/test split

Используем стратифицированное разделение с `random_state=42` для воспроизводимости результатов.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (32950, 19)
X_test: (8238, 19)
y_train: (32950,)
y_test: (8238,)


In [8]:
assert len(X_train) + len(X_test) == len(X)
assert len(y_train) + len(y_test) == len(y)

### 3.2. Проверка распределения классов

Проверим, сохранилось ли исходное соотношение классов после разделения.

In [9]:
class_distribution = pd.DataFrame({
    'Исходные данные': y.value_counts(normalize=True) * 100,
    'Train': y_train.value_counts(normalize=True) * 100,
    'Test': y_test.value_counts(normalize=True) * 100
})

class_distribution.round(2)

,Исходные данные,Train,Test
y,,,
0,88.73,88.73,88.74
1,11.27,11.27,11.26


## 4. Обработка особенностей данных

На основе результатов EDA определим способы обработки неизвестных и редких категорий, специальных значений и выбросов.

Все решения по предобработке будем принимать без использования тестовой выборки.

### 4.1. Неизвестные и редкие категории

Значения `unknown` сохраним как отдельные категории, поскольку отсутствие информации может быть полезно для классификации.

Редкие категории также оставим без изменений.

Для обработки категориальных признаков используем `OneHotEncoder` и `TargetEncoder`.

In [10]:
unknown_share = (X_train == 'unknown').mean() * 100

unknown_share[unknown_share > 0].round(2)

job           0.78
marital       0.21
education     4.22
default      20.84
housing       2.44
loan          2.44
dtype: float64

### 4.2. Обработка `pdays`

Значение `999` в признаке `pdays` обозначает отсутствие соответствующего предыдущего контакта.

Заменим исходный признак двумя переменными:

- `pdays_known` — наличие известного интервала с предыдущего контакта;
- `pdays_days` — количество дней, если интервал известен.

Для наблюдений с `pdays = 999` установим оба значения равными нулю.

In [11]:
def transform_pdays(X):
    X = X.copy()

    X['pdays_known'] = (X['pdays'] != 999).astype(int)

    X['pdays_days'] = X['pdays'].where(
        X['pdays'] != 999,
        0
    )

    return X.drop(columns='pdays')

In [12]:
sample = pd.DataFrame({
    'pdays': [999, 3, 10],
    'previous': [0, 1, 2]
})

transform_pdays(sample)

,previous,pdays_known,pdays_days
0,0,0,0
1,1,1,3
2,2,1,10


### Выводы

По результатам EDA выбраны следующие решения:

- `unknown` и редкие категории сохраняем без изменений.
- `pdays` преобразуем в два признака: `pdays_known` и `pdays_days`.
- Выбросы и сильно коррелирующие признаки пока сохраняем. Необходимость их обработки проверим при моделировании.

Выбранные преобразования реализуем в пайплайне предобработки.

## 5. Кодирование и масштабирование

Для подготовки данных используем несколько способов преобразования признаков:

- `OneHotEncoder` — для категориальных признаков с небольшим числом значений.
- `TargetEncoder` — для `job` и `education`, чтобы сократить размерность.
- Циклическое кодирование — для `month`, учитывая периодичность календарных месяцев.
- `StandardScaler` — для масштабирования числовых признаков.

Все преобразования будут выполняться внутри пайплайна предобработки.

### 5.1. Разделение признаков по типам

Разделим признаки на группы в зависимости от выбранного способа обработки.

При формировании списков учтём замену `pdays` на `pdays_days` и `pdays_known`.

In [13]:
# Числовые признаки
num_cols = X_train.select_dtypes(include='number').columns.tolist()

num_cols.remove('pdays')
num_cols.append('pdays_days')

# Категориальные признаки
cat_cols = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

# Группы для разных способов кодирования
target_cols = ['job', 'education']
cyclical_cols = ['month']

onehot_cols = [
    col for col in cat_cols
    if col not in target_cols + cyclical_cols
]

# Бинарные признаки без масштабирования
binary_cols = ['pdays_known']

In [14]:
feature_groups = {
    'Числовые': num_cols,
    'Target Encoding': target_cols,
    'Cyclical Encoding': cyclical_cols,
    'One-Hot Encoding': onehot_cols,
    'Бинарные': binary_cols
}

for group, columns in feature_groups.items():
    print(f'{group} ({len(columns)}): {columns}')

Числовые (9): ['age', 'campaign', 'previous', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'pdays_days']
Target Encoding (2): ['job', 'education']
Cyclical Encoding (1): ['month']
One-Hot Encoding (7): ['marital', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'poutcome']
Бинарные (1): ['pdays_known']


### 5.2. One-Hot Encoding

Для категориальных признаков с небольшим количеством значений используем `OneHotEncoder`.

Каждая категория будет представлена отдельным бинарным признаком. Значения `unknown` и редкие категории сохранятся.

Параметр `handle_unknown='ignore'` позволит обрабатывать категории, не встречавшиеся при обучении кодировщика.

In [15]:
onehot_transformer = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

### 5.3. Target Encoding

Для `job` и `education` используем `TargetEncoder`, заменяющий категории сглаженными оценками доли положительного класса.

Для уменьшения риска переобучения применим внутреннюю кросс-валидацию (`cv=5`). Это позволяет кодировать обучающие наблюдения без использования их собственных целевых значений.

Полученные числовые признаки дополнительно стандартизируем.

In [16]:
target_transformer = Pipeline([
    ('encoder', TargetEncoder(
        target_type='binary',
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        ),
        smooth='auto'
    )),
    ('scaler', StandardScaler())
])

### 5.4. Циклическое кодирование `month`

Признак `month` имеет циклическую структуру: после декабря снова наступает январь.

Преобразуем месяц в две числовые переменные с помощью синуса и косинуса. Это позволит учитывать календарную близость месяцев без создания отдельных бинарных столбцов.

Такое представление предполагает циклическую зависимость, поэтому его эффективность позднее сравним с One-Hot Encoding.

In [17]:
month_mapping = {
    'jan': 1,
    'feb': 2,
    'mar': 3,
    'apr': 4,
    'may': 5,
    'jun': 6,
    'jul': 7,
    'aug': 8,
    'sep': 9,
    'oct': 10,
    'nov': 11,
    'dec': 12
}

In [18]:
def transform_month(X):
    month_num = X['month'].map(month_mapping)

    if month_num.isna().any():
        raise ValueError('Обнаружены неизвестные значения month')

    return pd.DataFrame({
        'month_sin': np.sin(2 * np.pi * month_num / 12),
        'month_cos': np.cos(2 * np.pi * month_num / 12)
    }, index=X.index)

In [19]:
def month_feature_names(transformer, input_features):
    return np.array(['month_sin', 'month_cos'])

month_transformer = FunctionTransformer(
    transform_month,
    validate=False,
    feature_names_out=month_feature_names
)

### 5.5. Масштабирование числовых признаков

Для числовых признаков используем `StandardScaler`, который вычитает среднее и делит значения на стандартное отклонение.

Параметры стандартизации будут рассчитываться только на обучающих данных внутри пайплайна.

Бинарный `pdays_known` и циклические признаки `month_sin`, `month_cos` оставим без дополнительного масштабирования.

In [20]:
numeric_transformer = StandardScaler()

## 6. Построение пайплайна

Объединим выбранные методы предобработки в единый пайплайн.

Это позволит последовательно преобразовывать признаки и использовать одинаковую предобработку для Logistic Regression и SVM.

Параметры преобразований будут рассчитываться только на обучающих данных, в том числе внутри кросс-валидации.

### 6.1. Объединение преобразований через ColumnTransformer

Используем `ColumnTransformer` для применения разных методов предобработки к соответствующим группам признаков.

Числовые признаки стандартизируем, категориальные закодируем выбранными методами, а бинарный `pdays_known` оставим без изменений.

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, num_cols),
        ('onehot', onehot_transformer, onehot_cols),
        ('target', target_transformer, target_cols),
        ('month', month_transformer, cyclical_cols),
        ('binary', 'passthrough', binary_cols)
    ],
    remainder='drop',
    verbose_feature_names_out=True
)

### 6.2. Создание общего Pipeline

Создадим пайплайн, объединяющий преобразование `pdays` и последующую обработку всех признаков.

Сначала исходный `pdays` будет заменён на `pdays_known` и `pdays_days`, затем `ColumnTransformer` применит кодирование и масштабирование.

In [22]:
preprocessing_pipeline = Pipeline([
    ('pdays', FunctionTransformer(
        transform_pdays,
        validate=False
    )),
    ('columns', preprocessor)
])

### 6.3. Проверка работы пайплайна

Применим пайплайн к обучающей выборке и проверим размерность полученной матрицы, названия признаков и наличие некорректных значений.

Тестовую выборку на данном этапе не используем.

In [23]:
X_train_transformed = preprocessing_pipeline.fit_transform(
    X_train,
    y_train
)

In [24]:
print('До предобработки:', X_train.shape)
print('После предобработки:', X_train_transformed.shape)

До предобработки: (32950, 19)
После предобработки: (32950, 37)


In [25]:
feature_names = (
    preprocessing_pipeline
    .named_steps['columns']
    .get_feature_names_out()
)

X_train_prepared = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_train_prepared.head()

,numeric__age,numeric__campaign,numeric__previous,numeric__emp_var_rate,numeric__cons_price_idx,numeric__cons_conf_idx,numeric__euribor3m,numeric__nr_employed,numeric__pdays_days,onehot__marital_divorced,...,onehot__day_of_week_tue,onehot__day_of_week_wed,onehot__poutcome_failure,onehot__poutcome_nonexistent,onehot__poutcome_success,target__job,target__education,month__month_sin,month__month_cos,binary__pdays_known
27549,-0.964771,0.891970,-0.350073,-0.116686,-0.649341,-0.325891,0.286949,0.397226,-0.165648,0.0,...,0.0,1.0,0.0,1.0,0.0,-0.022073,1.112328,-0.500000,0.866025,0.0
27965,-0.197393,-0.202563,-0.350073,0.646775,0.722568,0.883719,0.712262,0.330917,-0.165648,0.0,...,0.0,0.0,0.0,1.0,0.0,0.387508,1.143506,0.500000,-0.866025,0.0
36497,1.049596,-0.567408,1.665464,-1.198255,-1.179790,-1.233098,-1.349182,-0.938621,-0.165648,0.0,...,0.0,1.0,1.0,0.0,0.0,-0.841948,-0.181111,0.500000,-0.866025,0.0
315,-0.293315,-0.567408,-0.350073,0.837640,-0.227747,0.948520,0.772774,0.843429,-0.165648,0.0,...,0.0,0.0,0.0,1.0,0.0,-0.090905,-0.032864,-0.866025,-0.500000,0.0
38191,0.282218,-0.567408,-0.350073,-0.116686,-0.649341,-0.325891,0.305967,0.397226,-0.165648,0.0,...,1.0,0.0,0.0,1.0,0.0,-0.052761,1.143506,-0.500000,0.866025,0.0


In [26]:
print('Размерность:', X_train_prepared.shape)

print(
    'Пропущенные значения:',
    X_train_prepared.isna().sum().sum()
)

print(
    'Бесконечные значения:',
    np.isinf(X_train_prepared.to_numpy()).sum()
)

Размерность: (32950, 37)
Пропущенные значения: 0
Бесконечные значения: 0


In [27]:
assert len(X_train_prepared) == len(X_train)
assert X_train_prepared.columns.is_unique
assert np.isfinite(X_train_prepared.to_numpy()).all()
assert X_train_prepared.index.equals(X_train.index)

## 7. Итоги предобработки

В ходе предобработки данные подготовлены для обучения и сравнения моделей Logistic Regression и SVM.

**Основные результаты:**

1. Из набора признаков исключён `duration`, поскольку его значение недоступно до завершения телефонного разговора.
2. Данные разделены на обучающую (80%) и тестовую (20%) выборки со стратификацией по целевой переменной.
3. Специальное значение `pdays = 999` обработано путём создания двух признаков: `pdays_known` и `pdays_days`.
4. Для категориальных признаков выбраны три метода кодирования:
   - **One-Hot Encoding** — для признаков с небольшим количеством категорий;
   - **Target Encoding** — для `job` и `education`, с внутренним кросс-фиттингом для снижения риска переобучения;
   - **Циклическое кодирование** — для `month`.
5. Числовые признаки стандартизированы с помощью `StandardScaler`. Редкие категории, потенциальные выбросы и коррелирующие признаки сохранены.

Все преобразования объединены в единый `Pipeline` с использованием `ColumnTransformer`.

**Результат:** после предобработки обучающая выборка содержит 32 950 наблюдений и 37 числовых признаков. Количество наблюдений сохранилось, пропущенные и бесконечные значения отсутствуют.

### Следующий этап

В `03_modeling.ipynb` будут обучены и сравнены три модели:

- Logistic Regression;
- Linear SVM;
- RBF SVM.

Предобработка будет выполняться внутри кросс-валидации для предотвращения утечки данных. Дополнительно планируется сравнить комбинированное кодирование с базовым One-Hot Encoding по ROC-AUC и PR-AUC.